# Flight Price Regression — Travel Analytics Capstone
**GitHub repository:** https://github.com/St-virus1210/travel-analytics-capstone

This notebook fits Ridge and Random Forest regression models to `flights.price`, evaluates MAE/RMSE/R² on a held-out split, inspects residuals and saves the selected model. Place the supplied CSVs in `data/` before running.

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA=Path('data'); flights=pd.read_csv(DATA/'flights.csv')
df=flights.copy(); df['price']=pd.to_numeric(df['price'],errors='coerce'); df=df.dropna(subset=['price'])
dt=pd.to_datetime(df['date'],errors='coerce'); df['date_year']=dt.dt.year.fillna(2000); df['date_month']=dt.dt.month.fillna(1); df['date_dayofweek']=dt.dt.dayofweek.fillna(0)
cat=['from','to','flightType','agency']; num=['time','distance','date_year','date_month','date_dayofweek']; X=df[cat+num].copy(); y=df['price'].astype(float)
for c in cat: X[c]=X[c].fillna('unknown').astype(str)
for c in num: X[c]=pd.to_numeric(X[c],errors='coerce')
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
prep=ColumnTransformer([('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=2))]),cat),('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num)])
models={'Ridge':Ridge(alpha=2.0),'Random Forest':RandomForestRegressor(n_estimators=100,min_samples_leaf=2,max_depth=24,random_state=42,n_jobs=-1)}
rows=[]; fitted={}
for name,est in models.items():
    pipe=Pipeline([('preprocess',prep),('model',est)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_test); fitted[name]=pipe
    rows.append({'model':name,'MAE':mean_absolute_error(y_test,pred),'RMSE':mean_squared_error(y_test,pred)**.5,'R2':r2_score(y_test,pred)})
results=pd.DataFrame(rows).sort_values('MAE'); display(results)
best_name=results.iloc[0]['model']; best=fitted[best_name]; print('Selected by holdout MAE:',best_name)
pred=best.predict(X_test); residual=y_test.to_numpy()-pred
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].scatter(pred[:2500],residual[:2500],alpha=.3); ax[0].axhline(0,ls='--'); ax[0].set_xlabel('Predicted price'); ax[0].set_ylabel('Residual'); ax[1].hist(residual,bins=40); plt.tight_layout(); plt.show()
Path('artifacts').mkdir(exist_ok=True); joblib.dump(best,'artifacts/flight_price_model.joblib'); Path('artifacts/flight_metrics_notebook.json').write_text(results.to_json(orient='records',indent=2)); print('Saved model and metrics.')

**Caution:** unusually high R² may indicate leakage or a random-split artifact. Validate chronologically and inspect target/feature dependence before claiming generalization. Price is reported in source dataset units; currency is unspecified.